# Sentiment Analysis and Classification of Bangla Text Data

This notebook performs sentiment analysis and classification on a Bangla text dataset. We will explore the data, preprocess it, generate visualizations, and use multiple machine learning models for classification. We will also implement a deep learning model for comparison.

# 1. Importing Libraries and Dataset

First, we import the necessary libraries for data manipulation, visualization, natural language processing, and machine learning. These libraries provide various functionalities such as reading the dataset, processing text data, and building machine learning models.


In [ ]:
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud


# 2. Downloading NLTK Resources

We need to download the necessary resources for tokenization using NLTK. Tokenization is the process of splitting text into individual words or tokens.

In [ ]:
# Download the necessary resources for tokenization
nltk.download('punkt')

# 3. Loading the Dataset

Next, we load the dataset from a CSV file. This dataset contains comments and their corresponding sentiments (positive or negative).

In [ ]:
# Load the dataset
df = pd.read_csv('/kaggle/input/revbangla/data.csv')

# 4. Plotting Distributions

We plot the distributions of sentiments and ratings to understand the balance and range of the data. This helps us visualize the proportion of positive and negative comments and the distribution of ratings.

In [ ]:
def plot_distribution(df, column, title, xlabel, ylabel):
    plt.figure(figsize=(10, 6))
    sns.countplot(x=column, data=df)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.show()

plot_distribution(df, 'sentiment', 'Distribution of Sentiments', 'Sentiment', 'Count')
plot_distribution(df, 'rating', 'Distribution of Ratings', 'Rating', 'Count')

# 5. Generating Word Clouds

We generate word clouds for positive and negative comments to visualize the most frequent words. This gives us an idea of the common words used in different sentiments.

In [ ]:
def generate_word_cloud(df, sentiment, title, font_path):
    comments = df[df['sentiment'] == sentiment]['comment']
    preprocessed_comments = comments.apply(lambda x: x.lower())
    word_corpus = ' '.join(preprocessed_comments)
    wc = WordCloud(
        width=800, height=400, background_color='white', colormap='plasma', font_path=font_path,
        regexp=r"[\u0980-\u09FF]+"
    ).generate(word_corpus)
    
    plt.figure(figsize=(10, 6))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(title)
    plt.show()

bangla_font_path = '/kaggle/input/bangla-font-kalpurush/kalpurush.ttf'
generate_word_cloud(df, 'positive', 'Word Cloud for Positive Comments', bangla_font_path)
generate_word_cloud(df, 'negative', 'Word Cloud for Negative Comments', bangla_font_path)

# 6. Data Preprocessing

We preprocess the data by removing duplicates and handling missing values. We also map sentiments to binary values (0 for negative and 1 for positive) and tokenize the comments.

In [ ]:
df = df[['comment', 'sentiment']].drop_duplicates().dropna()
df['sentiment'] = df['sentiment'].map({'negative': 0, 'positive': 1})
df['tokenized_comment'] = df['comment'].apply(word_tokenize)

# 7. Data Augmentation

We use oversampling and undersampling techniques to balance the dataset. Oversampling increases the number of minority class samples, while undersampling reduces the number of majority class samples.

In [ ]:
oversampler = RandomOverSampler(random_state=1337)
X_over, y_over = oversampler.fit_resample(df[['tokenized_comment']], df['sentiment'])

undersampler = RandomUnderSampler(random_state=1337)
X_augmented, y_augmented = undersampler.fit_resample(X_over, y_over)

# 8. Feature Extraction using TF-IDF

We extract features from the text using TF-IDF (Term Frequency-Inverse Document Frequency) with trigram features. This transforms the text data into numerical features that can be used by machine learning models.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_augmented['tokenized_comment'], y_augmented, test_size=0.2, random_state=1337)

tfidf = TfidfVectorizer(tokenizer=lambda doc: doc, lowercase=False, ngram_range=(1, 3))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# 9. Training and Evaluating Models

We train and evaluate multiple machine learning models, including Naive Bayes, K-Nearest Neighbors, Random Forest, Logistic Regression, Gradient Boosting, Decision Tree, LightGBM, XGBoost, MLP, and SVM. We print the performance metrics for each model.

In [ ]:
models = {
    'Naive Bayes': MultinomialNB(),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Random Forest': RandomForestClassifier(),
    'Logistic Regression': LogisticRegression(max_iter=200),
    'Gradient Boosting': GradientBoostingClassifier(),
    'Decision Tree': DecisionTreeClassifier(),
    'LightGBM': LGBMClassifier(),
    'XGBoost': XGBClassifier(eval_metric='mlogloss'),
    'MLP': MLPClassifier(hidden_layer_sizes=(100,), max_iter=200),
    'SVM': SVC(),
}

def evaluate_models(models, X_train, X_test, y_train, y_test):
    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        acc = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='macro')
        recall = recall_score(y_test, y_pred, average='macro')
        f1 = f1_score(y_test, y_pred, average='macro')
        
        print(f"Results for {model_name}:")
        print("\n")
        print("Accuracy:", acc)
        print("Precision:", precision)
        print("Recall:", recall)
        print("F1 Score:", f1)
        print("Classification Report:")
        print(classification_report(y_test, y_pred, digits=4))
        print("=" * 60)
        print("\n")

evaluate_models(models, X_train_tfidf, X_test_tfidf, y_train, y_test)

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

def plot_metrics(models, X_train, X_test, y_train, y_test):
    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Calculate performance metrics
        acc = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='macro')
        recall = recall_score(y_test, y_pred, average='macro')
        f1 = f1_score(y_test, y_pred, average='macro')

        
    # Plot accuracy, precision, recall, and F1 score
    metrics = {'Accuracy': [], 'Precision': [], 'Recall': [], 'F1 Score': []}
    
    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        acc = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='macro')
        recall = recall_score(y_test, y_pred, average='macro')
        f1 = f1_score(y_test, y_pred, average='macro')
        
        metrics['Accuracy'].append(acc)
        metrics['Precision'].append(precision)
        metrics['Recall'].append(recall)
        metrics['F1 Score'].append(f1)
    
    plt.figure(figsize=(10, 8))
    plt.barh(list(models.keys()), metrics['Accuracy'], color='royalblue')
    plt.xlabel('Accuracy')
    plt.title('Model Accuracy')
    plt.show()
    
    plt.figure(figsize=(10, 8))
    plt.barh(list(models.keys()), metrics['Precision'], color='green')
    plt.xlabel('Precision')
    plt.title('Model Precision')
    plt.show()
    
    plt.figure(figsize=(10, 8))
    plt.barh(list(models.keys()), metrics['Recall'], color='orange')
    plt.xlabel('Recall')
    plt.title('Model Recall')
    plt.show()
    
    plt.figure(figsize=(10, 8))
    plt.barh(list(models.keys()), metrics['F1 Score'], color='red')
    plt.xlabel('F1 Score')
    plt.title('Model F1 Score')
    plt.show()

plot_metrics(models, X_train_tfidf, X_test_tfidf, y_train, y_test)

# 10. Deep Learning Model

We build and train a simple neural network model for comparison with the machine learning models. We normalize the TF-IDF features and convert the labels to one-hot encoding before training the model.

In [ ]:
X_train_dense = X_train_tfidf.toarray()
X_test_dense = X_test_tfidf.toarray()
X_train_normalized = X_train_dense / X_train_dense.max()
X_test_normalized = X_test_dense / X_test_dense.max()

y_train_onehot = tf.keras.utils.to_categorical(y_train)
y_test_onehot = tf.keras.utils.to_categorical(y_test)

dl_model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_normalized.shape[1],)),
    Dense(64, activation='relu'),
    Dense(len(set(y_train)), activation='softmax')
])

dl_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

dl_model.fit(X_train_normalized, y_train_onehot, epochs=10, batch_size=32, validation_split=0.2)

y_pred_onehot = dl_model.predict(X_test_normalized)
y_pred = y_pred_onehot.argmax(axis=1)

acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')

print("Results for Deep Learning Model:")
print("\n")
print("Accuracy:", acc)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("Classification Report:")
print(classification_report(y_test, y_pred, digits=4))
print("=" * 60)